# Notebook 21b: Population Density Reproducibility Validation

**Date**: 2026-01-12  
**Goal**: Validate reproducibility on population density task (parallel to NB21 elevation)

**Motivation**:
- Population is DIFFERENT task from elevation (smoother, more concentrated)
- Multiple native resolutions available (1°, 30', 15', 2.5', 30")
- Cross-validate elevation findings
- NB19b showed ReLU beat splines on population - is it reproducible?

---

## Experiments

### Experiment 1: Global Multi-Seed (15' Resolution)
- **Config**: 15K samples, 15 arc-min resolution, 10 seeds
- **Question**: Is NB19b's ReLU advantage reproducible?
- **Baseline**: NB19b showed ReLU -0.64% (spline worse)

### Experiment 2: Sample Size Sensitivity
- **Sample sizes**: 5K, 10K, 20K, 50K
- **Resolution**: 15 arc-min
- **Question**: At what N do population results stabilize?

### Experiment 3: Resolution Comparison (Multi-Seed)
- **Resolutions**: 30_min (~55km), 15_min (~30km), 2pt5_min (~5km), 30_sec (~1km)
- **Seeds**: 10 runs each
- **Question**: Does resolution affect activation performance?
- **Hypothesis**: Finer resolution (more detail) → splines might help
- **Critical**: Use NATIVE resolutions, not resampled

### Experiment 4: Urban vs Rural Regions
- **Dense urban**: NYC, Tokyo regions (>1000 people/km²)
- **Rural**: Great Plains, Siberia (<100 people/km²)
- **Seeds**: 10 runs each
- **Question**: Do sharp urban boundaries favor splines?

### Experiment 5: Extended Training
- **Epochs**: 200 (vs 100)
- **Seeds**: 5 runs
- **Question**: Convergence validation

---

## Key Differences from NB21 (Elevation)

| Aspect | NB21 (Elevation) | NB21b (Population) |
|--------|------------------|--------------------|
| **Data** | ETOPO 60s (~2km) | GPW multiple resolutions |
| **Task** | Continuous elevation | Log population density |
| **Spatial pattern** | Mountainous features | Urban clusters |
| **Frequency content** | Multi-scale (mountains) | Sharp edges (cities) |
| **NB19 result** | Spline +0.36% | Spline -0.64% (worse) |

**If NB21 and NB21b give DIFFERENT results** (e.g., splines help for elevation but not population):
- ✅ Strong evidence for task-dependent effects
- Publication: "Task-Specific Learned Activation Performance"

**If both show SAME pattern**:
- ✅ Generalizable finding
- Either "ReLU always wins" or "Splines always help"

---

## Computational Budget

**Exp 1**: 10 seeds × 3 acts × 90s = ~45 min  
**Exp 2**: 4 sizes × 10 seeds × 3 acts × 60-120s = ~4 hours  
**Exp 3**: 4 resolutions × 10 seeds × 3 acts × 90-150s = ~5 hours (30_sec is larger)  
**Exp 4**: 2 regions × 10 seeds × 3 acts × 90s = ~90 min  
**Exp 5**: 5 seeds × 3 acts × 180s = ~45 min

**Total**: ~12 hours (can run in parallel with NB21)

---

## Success Criteria

Same as NB21:
1. Std < 2% of mean across 10 seeds
2. 95% CI excludes zero if claiming advantage
3. Consistent across sample sizes ≥20K

---
## Setup

In [1]:
# Environment setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

Cloning into 'satclip'...
remote: Enumerating objects: 628, done.
remote: Counting objects: 100% (307/307), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 628 (delta 223), reused 213 (delta 164), pack-reused 321 (from 2)
Receiving objects: 100% (628/628), 82.87 MiB | 35.77 MiB/s, done.
Resolving deltas: 100% (326/326), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.9 MB/s eta 0:00:00
   ━━━

In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
from scipy import stats
import rasterio
import zipfile
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

import positional_encoding as PE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


---
## Data Loading (GPW Population)

In [3]:
# === COLAB SETUP: GPW Population Data ===
if 'COLAB_GPU' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')

    GPW_DIR = './gpw_data'
    os.makedirs(GPW_DIR, exist_ok=True)

    SOURCE_ZIP_PATH = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'

    print("Step 1: Extracting main archive...")
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as z:
        z.extractall(GPW_DIR)

    print("Step 2: Extracting nested TIF zips...")
    # Extract ALL resolutions for comprehensive testing
    RESOLUTIONS_TO_EXTRACT = ['30_min', '15_min', '2pt5_min', '30_sec']
    YEAR = 2020

    extracted = 0
    for res in RESOLUTIONS_TO_EXTRACT:
        zip_name = f"gpw-v4-population-density-rev11_{YEAR}_{res}_tif.zip"
        zip_path = os.path.join(GPW_DIR, zip_name)

        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(GPW_DIR)
            extracted += 1
            print(f"  ✓ Extracted {zip_name}")
        else:
            print(f"  ✗ Not found: {zip_name}")

    print(f"\nStep 3: Verifying TIF files...")
    tif_files = [f for f in os.listdir(GPW_DIR) if f.endswith('.tif')]
    print(f"  Found {len(tif_files)} TIF files:")
    for f in sorted(tif_files):
        size_mb = os.path.getsize(os.path.join(GPW_DIR, f)) / 1e6
        print(f"    {f} ({size_mb:.1f} MB)")

    print("\n✅ Setup complete!")
else:
    GPW_DIR = './gpw_data'
    print("Running locally - ensure GPW data is in ./gpw_data/")

Mounted at /content/drive
Step 1: Extracting main archive...
Step 2: Extracting nested TIF zips...
  ✓ Extracted gpw-v4-population-density-rev11_2020_30_min_tif.zip
  ✓ Extracted gpw-v4-population-density-rev11_2020_15_min_tif.zip
  ✓ Extracted gpw-v4-population-density-rev11_2020_2pt5_min_tif.zip
  ✓ Extracted gpw-v4-population-density-rev11_2020_30_sec_tif.zip

Step 3: Verifying TIF files...
  Found 4 TIF files:
    gpw_v4_population_density_rev11_2020_15_min.tif (1.1 MB)
    gpw_v4_population_density_rev11_2020_2pt5_min.tif (24.8 MB)
    gpw_v4_population_density_rev11_2020_30_min.tif (0.3 MB)
    gpw_v4_population_density_rev11_2020_30_sec.tif (369.7 MB)

✅ Setup complete!


In [4]:
def load_population_data(resolution='15_min'):
    """
    Load GPW population density at specified resolution.

    Available resolutions:
    - '30_min': ~55km at equator (720 × 360 grid)
    - '15_min': ~30km at equator (1440 × 720 grid) [NB19b baseline]
    - '2pt5_min': ~5km at equator (8640 × 4320 grid)
    - '30_sec': ~1km at equator (43200 × 21600 grid)
    """
    # Actual filenames use underscores, not hyphens (e.g., gpw_v4 not gpw-v4)
    tif_filename = f'gpw_v4_population_density_rev11_2020_{resolution}.tif'
    tif_path = os.path.join(GPW_DIR, tif_filename)

    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"TIF file not found: {tif_filename}\nChecked path: {tif_path}")

    print(f"Loading: {os.path.basename(tif_path)}")

    with rasterio.open(tif_path) as src:
        data = src.read(1)  # Read first (only) band
        transform = src.transform

        # Get coordinate arrays
        height, width = data.shape

        # Rasterio uses (col, row) indexing
        lons = np.array([transform * (i, 0) for i in range(width)])[:, 0]
        lats = np.array([transform * (0, j) for j in range(height)])[:, 1]

        # Handle nodata
        nodata = src.nodata
        if nodata is not None:
            data = np.where(data == nodata, -9999, data)

        # Set negative/zero to small positive for log transform
        data = np.where(data <= 0, 1e-6, data)

    print(f"  Shape: {data.shape}")
    print(f"  Lat range: [{lats.min():.2f}, {lats.max():.2f}]")
    print(f"  Lon range: [{lons.min():.2f}, {lons.max():.2f}]")
    print(f"  Density range: [{data[data > 0].min():.2e}, {data.max():.2e}] people/km²")
    print(f"  Valid pixels: {(data > 0).sum():,} / {data.size:,}")

    return data, lons, lats, resolution


# Load default resolution for initial experiments
print("="*70)
print("LOADING GPW POPULATION DENSITY DATA")
print("="*70)

population, lons, lats, resolution = load_population_data('15_min')

print("\n✅ Population data loaded")
print("="*70)

LOADING GPW POPULATION DENSITY DATA
Loading: gpw_v4_population_density_rev11_2020_15_min.tif
  Shape: (720, 1440)
  Lat range: [-89.75, 90.00]
  Lon range: [-180.00, 179.75]
  Density range: [1.92e-09, 2.98e+04] people/km²
  Valid pixels: 1,036,800 / 1,036,800

✅ Population data loaded


---
## Model Definitions (Same as NB21)

In [5]:
class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class SirenLayer(nn.Module):
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class UniversalEncoder(nn.Module):
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:
            x = self.posenc(coords)

        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


print("✅ Model classes loaded")

✅ Model classes loaded


---
## Training Utilities

In [6]:
def sample_global_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample globally with spatial blocking.
    For population, excludes zero/nodata pixels.
    """
    np.random.seed(seed)

    # Valid data mask (population > 0)
    valid = data > 0

    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lons, lats)

    # Flatten
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = data[valid]

    # Sample subset
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals

    # Spatial blocking
    n_lon_cells = int(360 / grid_size)
    n_lat_cells = int(180 / grid_size)
    n_cells = n_lon_cells * n_lat_cells

    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon + 180) / grid_size)
        lat_cell = int((lat + 90) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)

    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)

    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def sample_regional_blocked(data, lons, lats, region_bounds, n_samples=5000,
                           grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample from a regional subset with spatial blocking.
    region_bounds: dict with 'lat_min', 'lat_max', 'lon_min', 'lon_max'
    """
    np.random.seed(seed)

    # Find indices for region
    lat_mask = (lats >= region_bounds['lat_min']) & (lats <= region_bounds['lat_max'])
    lon_mask = (lons >= region_bounds['lon_min']) & (lons <= region_bounds['lon_max'])

    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]

    regional_data = data[np.ix_(lat_idx, lon_idx)]
    regional_lats = lats[lat_idx]
    regional_lons = lons[lon_idx]

    lon_grid, lat_grid = np.meshgrid(regional_lons, regional_lats)
    valid = regional_data > 0

    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = regional_data[valid]

    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals

    region_width = region_bounds['lon_max'] - region_bounds['lon_min']
    region_height = region_bounds['lat_max'] - region_bounds['lat_min']

    n_lon_cells = max(1, int(region_width / grid_size))
    n_lat_cells = max(1, int(region_height / grid_size))
    n_cells = n_lon_cells * n_lat_cells

    test_cells = set(np.random.choice(n_cells, max(1, int(n_cells * test_ratio)), replace=False))

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon - region_bounds['lon_min']) / grid_size)
        lat_cell = int((lat - region_bounds['lat_min']) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)

    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)

    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def train_population_model(name, encoder, coords_train, vals_train, coords_test, vals_test,
                          epochs=100, batch_size=256, lr=1e-3, verbose=False, track_epochs=False):
    """
    Train population prediction model.
    Uses log(population + 1) as target.
    """
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # Target: log(population + 1)
    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test), dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_r2 = -float('inf')
    r2_history = [] if track_epochs else None
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        # Evaluate every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1 or track_epochs:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)

            if track_epochs:
                r2_history.append(r2)

            if verbose and (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    result = {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
    }

    if track_epochs:
        result['r2_history'] = r2_history

    return result


print("✅ Training utilities loaded")

✅ Training utilities loaded


---
## Statistics Utilities (Same as NB21)

In [7]:
def compute_stats(values):
    """Compute comprehensive statistics."""
    values = np.array(values)
    n = len(values)
    mean = values.mean()
    std = values.std(ddof=1) if n > 1 else 0.0
    stderr = std / np.sqrt(n) if n > 1 else 0.0

    if n > 1:
        ci = stats.t.interval(0.95, n-1, loc=mean, scale=stderr)
    else:
        ci = (mean, mean)

    return {
        'n': n,
        'mean': mean,
        'std': std,
        'stderr': stderr,
        'min': values.min(),
        'max': values.max(),
        'ci_low': ci[0],
        'ci_high': ci[1],
        'cv': (std / mean * 100) if mean != 0 else 0.0,
    }


def print_stats(label, stats_dict):
    """Pretty print statistics."""
    print(f"{label}:")
    print(f"  Mean ± Std:  {stats_dict['mean']:.4f} ± {stats_dict['std']:.4f}")
    print(f"  Range:       [{stats_dict['min']:.4f}, {stats_dict['max']:.4f}]")
    print(f"  95% CI:      [{stats_dict['ci_low']:.4f}, {stats_dict['ci_high']:.4f}]")
    print(f"  CV:          {stats_dict['cv']:.2f}%")
    print(f"  N:           {stats_dict['n']}")


def compare_activations(relu_r2s, spline_r2s):
    """Compare ReLU vs Spline with statistical tests."""
    relu_r2s = np.array(relu_r2s)
    spline_r2s = np.array(spline_r2s)

    if len(relu_r2s) == len(spline_r2s) and len(relu_r2s) > 1:
        t_stat, p_value = stats.ttest_rel(spline_r2s, relu_r2s)
    else:
        t_stat, p_value = np.nan, np.nan

    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)

    print("\n" + "="*60)
    print("SPLINE vs RELU COMPARISON")
    print("="*60)

    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)

    print("\nReLU:")
    print(f"  Mean ± Std:  {relu_stats['mean']:.4f} ± {relu_stats['std']:.4f}")
    print(f"  95% CI:      [{relu_stats['ci_low']:.4f}, {relu_stats['ci_high']:.4f}]")

    print("\nSpline:")
    print(f"  Mean ± Std:  {spline_stats['mean']:.4f} ± {spline_stats['std']:.4f}")
    print(f"  95% CI:      [{spline_stats['ci_low']:.4f}, {spline_stats['ci_high']:.4f}]")

    print("\nSpline Advantage (%):")
    print(f"  Mean ± Std:  {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%")
    print(f"  Range:       [{adv_stats['min']:+.2f}%, {adv_stats['max']:+.2f}%]")
    print(f"  95% CI:      [{adv_stats['ci_low']:+.2f}%, {adv_stats['ci_high']:+.2f}%]")

    if not np.isnan(p_value):
        print(f"\nPaired t-test: t={t_stat:.3f}, p={p_value:.4f}")
        if p_value < 0.05:
            if adv_stats['mean'] > 0:
                print("  ✅ SIGNIFICANT: Spline wins (p < 0.05)")
            else:
                print("  ✅ SIGNIFICANT: ReLU wins (p < 0.05)")
        else:
            print("  ❌ NOT SIGNIFICANT: No clear winner (p ≥ 0.05)")

    if adv_stats['ci_low'] > 1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, advantage > 1%")
    elif adv_stats['ci_high'] < -1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, disadvantage > 1%")
    else:
        print("\n  ⚠️  NO PRACTICAL SIGNIFICANCE: 95% CI includes small effects")

    print("="*60)

    return adv_stats


print("✅ Statistics utilities loaded")

✅ Statistics utilities loaded


---
## Experiment 1: Global Multi-Seed (15' Resolution)

**Baseline**: NB19b showed ReLU beat Spline by -0.64%

In [8]:
print("="*80)
print("EXPERIMENT 1: GLOBAL MULTI-SEED VALIDATION (Population 15')")
print("="*80)

SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline', 'siren']
N_SAMPLES = 15000
EPOCHS = 100

results_exp1 = []

for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"Seed {seed} ({seed-41}/10)")
    print("="*80)

    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        population, lons, lats, n_samples=N_SAMPLES, seed=seed
    )
    print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")
    print(f"Population range: [{vals_train.min():.2e}, {vals_train.max():.2e}] people/km²")

    for act in ACTIVATIONS:
        print(f"\n  {act.upper()}...", end=" ")

        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_population_model(
            f'pop_global_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            epochs=EPOCHS,
            verbose=False
        )

        res['seed'] = seed
        res['activation'] = act
        res['n_samples'] = N_SAMPLES

        results_exp1.append(res)
        print(f"R²={res['r2']:.4f}, Time={res['time']:.1f}s")

df_exp1 = pd.DataFrame(results_exp1)

print("\n" + "="*80)
print("EXPERIMENT 1 SUMMARY")
print("="*80)
print(df_exp1[['seed', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

EXPERIMENT 1: GLOBAL MULTI-SEED VALIDATION (Population 15')

Seed 42 (1/10)
Samples: 10512 train, 4488 test
Population range: [1.27e-07, 9.03e+03] people/km²

  RELU... R²=0.5563, Time=78.9s

  SPLINE... R²=0.5588, Time=108.5s

  SIREN... R²=0.5393, Time=79.3s

Seed 43 (2/10)
Samples: 10486 train, 4514 test
Population range: [2.09e-07, 1.43e+04] people/km²

  RELU... R²=0.6171, Time=76.7s

  SPLINE... R²=0.6196, Time=105.6s

  SIREN... R²=0.5818, Time=77.8s

Seed 44 (3/10)
Samples: 10592 train, 4408 test
Population range: [1.00e-06, 6.15e+03] people/km²

  RELU... R²=0.5363, Time=78.5s

  SPLINE... R²=0.5421, Time=108.5s

  SIREN... R²=0.5145, Time=79.5s

Seed 45 (4/10)
Samples: 10519 train, 4481 test
Population range: [2.12e-08, 3.44e+03] people/km²

  RELU... R²=0.6100, Time=78.7s

  SPLINE... R²=0.6125, Time=108.9s

  SIREN... R²=0.5544, Time=79.5s

Seed 46 (5/10)
Samples: 10513 train, 4487 test
Population range: [1.47e-07, 1.12e+04] people/km²

  RELU... R²=0.5985, Time=78.7s

  SP

In [9]:
# Statistical analysis
print("\n" + "="*80)
print("EXPERIMENT 1: STATISTICAL ANALYSIS")
print("="*80)

for act in ACTIVATIONS:
    act_data = df_exp1[df_exp1['activation'] == act]
    r2_values = act_data['r2'].values
    stats_dict = compute_stats(r2_values)
    print(f"\n{act.upper()}:")
    print_stats(f"  R² Statistics", stats_dict)

relu_r2s = df_exp1[df_exp1['activation'] == 'relu']['r2'].values
spline_r2s = df_exp1[df_exp1['activation'] == 'spline']['r2'].values

adv_stats = compare_activations(relu_r2s, spline_r2s)

print("\n" + "="*80)
print("COMPARISON TO NB19b (Single Seed)")
print("="*80)
print("\nNB19b Results (Regression):")
print("  ReLU:   R² = 0.7429")
print("  Spline: R² = 0.7381")
print("  Advantage: -0.64% (ReLU wins)")
print("\nNB21b Results (10 seeds):")
print(f"  ReLU:   R² = {relu_r2s.mean():.4f} ± {relu_r2s.std():.4f}")
print(f"  Spline: R² = {spline_r2s.mean():.4f} ± {spline_r2s.std():.4f}")
print(f"  Advantage: {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%")

if adv_stats['cv'] < 50:
    print("\n✅ STABLE: Results reproducible (CV < 50%)")
else:
    print("\n❌ UNSTABLE: High variance (CV ≥ 50%)")

print("="*80)


EXPERIMENT 1: STATISTICAL ANALYSIS

RELU:
  R² Statistics:
  Mean ± Std:  0.5904 ± 0.0316
  Range:       [0.5363, 0.6456]
  95% CI:      [0.5678, 0.6130]
  CV:          5.35%
  N:           10

SPLINE:
  R² Statistics:
  Mean ± Std:  0.5888 ± 0.0297
  Range:       [0.5421, 0.6277]
  95% CI:      [0.5676, 0.6101]
  CV:          5.04%
  N:           10

SIREN:
  R² Statistics:
  Mean ± Std:  0.5536 ± 0.0245
  Range:       [0.5145, 0.5961]
  95% CI:      [0.5361, 0.5712]
  CV:          4.43%
  N:           10

SPLINE vs RELU COMPARISON

ReLU:
  Mean ± Std:  0.5904 ± 0.0316
  95% CI:      [0.5678, 0.6130]

Spline:
  Mean ± Std:  0.5888 ± 0.0297
  95% CI:      [0.5676, 0.6101]

Spline Advantage (%):
  Mean ± Std:  -0.23% ± 2.18%
  Range:       [-3.54%, +2.28%]
  95% CI:      [-1.79%, +1.32%]

Paired t-test: t=-0.389, p=0.7065
  ❌ NOT SIGNIFICANT: No clear winner (p ≥ 0.05)

  ⚠️  NO PRACTICAL SIGNIFICANCE: 95% CI includes small effects

COMPARISON TO NB19b (Single Seed)

NB19b Results (Reg

---
## Experiment 2: Sample Size Sensitivity

Same as NB21 but for population

In [10]:
print("="*80)
print("EXPERIMENT 2: SAMPLE SIZE SENSITIVITY (Population)")
print("="*80)

SAMPLE_SIZES = [5000, 10000, 20000, 50000]
SEEDS = range(42, 52)
ACTIVATIONS = ['relu', 'spline']

results_exp2 = []

for n_samples in SAMPLE_SIZES:
    print(f"\n{'='*80}")
    print(f"Sample Size: {n_samples:,}")
    print("="*80)

    for seed in SEEDS:
        print(f"\n  Seed {seed} ({seed-41}/10)", end=" ")

        coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
            population, lons, lats, n_samples=n_samples, seed=seed
        )

        for act in ACTIVATIONS:
            print(f"{act.upper()}", end=" ")

            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )

            res = train_population_model(
                f'pop_n{n_samples}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                epochs=EPOCHS,
                verbose=False
            )

            res['seed'] = seed
            res['activation'] = act
            res['n_samples'] = n_samples

            results_exp2.append(res)
            print(f"({res['r2']:.4f})", end=" ")

        print()

df_exp2 = pd.DataFrame(results_exp2)

print("\n" + "="*80)
print("EXPERIMENT 2: VARIANCE vs SAMPLE SIZE")
print("="*80)

print("\n{:>10s} {:>15s} {:>15s} {:>10s} {:>10s}".format(
    "N Samples", "ReLU R²", "Spline R²", "Adv (%)", "Adv CV"
))
print("-" * 80)

for n_samples in SAMPLE_SIZES:
    subset = df_exp2[df_exp2['n_samples'] == n_samples]
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values

    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)

    print("{:>10,d} {:>15s} {:>15s} {:>10.2f} {:>10.2f}".format(
        n_samples,
        f"{relu_stats['mean']:.4f}±{relu_stats['std']:.4f}",
        f"{spline_stats['mean']:.4f}±{spline_stats['std']:.4f}",
        adv_stats['mean'],
        adv_stats['cv']
    ))

print("="*80)

EXPERIMENT 2: SAMPLE SIZE SENSITIVITY (Population)

Sample Size: 5,000

  Seed 42 (1/10) RELU (0.4830) SPLINE (0.4659) 

  Seed 43 (2/10) RELU (0.4847) SPLINE (0.4897) 

  Seed 44 (3/10) RELU (0.5111) SPLINE (0.4815) 

  Seed 45 (4/10) RELU (0.4446) SPLINE (0.4817) 

  Seed 46 (5/10) RELU (0.4550) SPLINE (0.5000) 

  Seed 47 (6/10) RELU (0.5509) SPLINE (0.5842) 

  Seed 48 (7/10) RELU (0.4591) SPLINE (0.5023) 

  Seed 49 (8/10) RELU (0.5799) SPLINE (0.5683) 

  Seed 50 (9/10) RELU (0.3698) SPLINE (0.3530) 

  Seed 51 (10/10) RELU (0.4041) SPLINE (0.4177) 

Sample Size: 10,000

  Seed 42 (1/10) RELU (0.5385) SPLINE (0.5437) 

  Seed 43 (2/10) RELU (0.5637) SPLINE (0.5285) 

  Seed 44 (3/10) RELU (0.5225) SPLINE (0.5093) 

  Seed 45 (4/10) RELU (0.5859) SPLINE (0.5719) 

  Seed 46 (5/10) RELU (0.5361) SPLINE (0.5295) 

  Seed 47 (6/10) RELU (0.6069) SPLINE (0.5874) 

  Seed 48 (7/10) RELU (0.5912) SPLINE (0.5632) 

  Seed 49 (8/10) RELU (0.6095) SPLINE (0.5518) 

  Seed 50 (9/10) RELU (0

---
## Experiment 3: Resolution Comparison

**CRITICAL**: Tests native resolutions, not resampled data

**Hypothesis**: Finer resolution (~1km) has sharp urban boundaries → splines might help

In [11]:
print("="*80)
print("EXPERIMENT 3: RESOLUTION COMPARISON (Multi-Seed)")
print("="*80)

# Test multiple native resolutions
RESOLUTIONS = ['30_min', '15_min', '2pt5_min', '30_sec']
RESOLUTION_INFO = {
    '30_min': {'name': '30 arc-min', 'km': 55, 'samples': 15000},
    '15_min': {'name': '15 arc-min', 'km': 30, 'samples': 15000},
    '2pt5_min': {'name': '2.5 arc-min', 'km': 5, 'samples': 15000},
    '30_sec': {'name': '30 arc-sec', 'km': 1, 'samples': 15000},  # May need to reduce if too slow
}

SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline']

results_exp3 = []

for res in RESOLUTIONS:
    print(f"\n{'='*80}")
    print(f"Resolution: {RESOLUTION_INFO[res]['name']} (~{RESOLUTION_INFO[res]['km']}km)")
    print("="*80)

    # Load this resolution
    pop_data, pop_lons, pop_lats, _ = load_population_data(res)
    n_samples = RESOLUTION_INFO[res]['samples']

    for seed in SEEDS:
        print(f"\n  Seed {seed}: ", end="")

        coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
            pop_data, pop_lons, pop_lats, n_samples=n_samples, seed=seed
        )

        for act in ACTIVATIONS:
            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )

            res_dict = train_population_model(
                f'pop_res{res}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                epochs=EPOCHS,
                verbose=False
            )

            res_dict['seed'] = seed
            res_dict['activation'] = act
            res_dict['resolution'] = res
            res_dict['res_km'] = RESOLUTION_INFO[res]['km']

            results_exp3.append(res_dict)
            print(f"{act}={res_dict['r2']:.4f} ", end="")

        print()

df_exp3 = pd.DataFrame(results_exp3)

print("\n" + "="*80)
print("EXPERIMENT 3: RESOLUTION EFFECT ANALYSIS")
print("="*80)

for res in RESOLUTIONS:
    subset = df_exp3[df_exp3['resolution'] == res]

    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values

    print(f"\n--- {RESOLUTION_INFO[res]['name']} (~{RESOLUTION_INFO[res]['km']}km) ---")
    compare_activations(relu_r2s, spline_r2s)

print("\n" + "="*80)
print("RESOLUTION TREND SUMMARY")
print("="*80)

print("\n{:>15s} {:>10s} {:>15s} {:>10s}".format(
    "Resolution", "~km", "Spline Adv (%)", "Adv CV (%)"
))
print("-" * 60)

for res in RESOLUTIONS:
    subset = df_exp3[df_exp3['resolution'] == res]
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)

    print("{:>15s} {:>10d} {:>15s} {:>10.2f}".format(
        RESOLUTION_INFO[res]['name'],
        RESOLUTION_INFO[res]['km'],
        f"{adv_stats['mean']:+.2f}±{adv_stats['std']:.2f}",
        adv_stats['cv']
    ))

print("\n" + "="*80)
print("INTERPRETATION:")
print("  - If advantage increases with finer resolution: splines capture detail")
print("  - If constant across resolutions: resolution doesn't matter")
print("  - Compare to NB21 (elevation): same pattern or different?")
print("="*80)

EXPERIMENT 3: RESOLUTION COMPARISON (Multi-Seed)

Resolution: 30 arc-min (~55km)
Loading: gpw_v4_population_density_rev11_2020_30_min.tif
  Shape: (360, 720)
  Lat range: [-89.50, 90.00]
  Lon range: [-180.00, 179.50]
  Density range: [6.59e-10, 2.24e+04] people/km²
  Valid pixels: 259,200 / 259,200

  Seed 42: relu=0.5789 spline=0.5498 

  Seed 43: relu=0.6292 spline=0.6132 

  Seed 44: relu=0.5946 spline=0.5995 

  Seed 45: relu=0.6221 spline=0.6192 

  Seed 46: relu=0.5502 spline=0.5600 

  Seed 47: relu=0.6277 spline=0.6102 

  Seed 48: relu=0.6531 spline=0.6474 

  Seed 49: relu=0.5487 spline=0.5340 

  Seed 50: relu=0.5677 spline=0.5661 

  Seed 51: relu=0.5650 spline=0.5688 

Resolution: 15 arc-min (~30km)
Loading: gpw_v4_population_density_rev11_2020_15_min.tif
  Shape: (720, 1440)
  Lat range: [-89.75, 90.00]
  Lon range: [-180.00, 179.75]
  Density range: [1.92e-09, 2.98e+04] people/km²
  Valid pixels: 1,036,800 / 1,036,800

  Seed 42: relu=0.5647 spline=0.5443 

  Seed 43: r

---
## Experiment 4: Urban vs Rural Regions

**Test if sharp urban boundaries favor splines**

In [12]:
print("="*80)
print("EXPERIMENT 4: URBAN vs RURAL REGIONS")
print("="*80)

# Define urban and rural regions
REGIONS = {
    'urban_east_us': {
        'lat_min': 35, 'lat_max': 45,
        'lon_min': -80, 'lon_max': -70,
        'type': 'urban',
        'description': 'US East Coast (NYC, Boston, DC)'
    },
    'rural_great_plains': {
        'lat_min': 38, 'lat_max': 48,
        'lon_min': -105, 'lon_max': -95,
        'type': 'rural',
        'description': 'US Great Plains (sparse population)'
    },
}

SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline']
N_SAMPLES = 10000  # Per region

results_exp4 = []

for region_name, region_bounds in REGIONS.items():
    print(f"\n{'='*80}")
    print(f"Region: {region_name.upper()} ({region_bounds['type']})")
    print(f"Description: {region_bounds['description']}")
    print("="*80)

    for seed in SEEDS:
        print(f"  Seed {seed}: ", end="")

        coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
            population, lons, lats, region_bounds, n_samples=N_SAMPLES, seed=seed
        )

        print(f"Pop range [{vals_train.min():.1f}, {vals_train.max():.1f}] ", end="")

        for act in ACTIVATIONS:
            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )

            res = train_population_model(
                f'pop_{region_name}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                epochs=EPOCHS,
                verbose=False
            )

            res['seed'] = seed
            res['activation'] = act
            res['region'] = region_name
            res['region_type'] = region_bounds['type']

            results_exp4.append(res)
            print(f"{act}={res['r2']:.4f} ", end="")

        print()

df_exp4 = pd.DataFrame(results_exp4)

print("\n" + "="*80)
print("EXPERIMENT 4: URBAN vs RURAL COMPARISON")
print("="*80)

for region_name in REGIONS.keys():
    subset = df_exp4[df_exp4['region'] == region_name]
    region_type = subset['region_type'].iloc[0]

    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values

    print(f"\n--- {region_name.upper()} ({region_type}) ---")
    compare_activations(relu_r2s, spline_r2s)

# Compare urban vs rural
urban_data = df_exp4[df_exp4['region_type'] == 'urban']
rural_data = df_exp4[df_exp4['region_type'] == 'rural']

urban_relu = urban_data[urban_data['activation'] == 'relu']['r2'].values
urban_spline = urban_data[urban_data['activation'] == 'spline']['r2'].values
urban_adv = 100 * (urban_spline - urban_relu) / urban_relu

rural_relu = rural_data[rural_data['activation'] == 'relu']['r2'].values
rural_spline = rural_data[rural_data['activation'] == 'spline']['r2'].values
rural_adv = 100 * (rural_spline - rural_relu) / rural_relu

urban_stats = compute_stats(urban_adv)
rural_stats = compute_stats(rural_adv)

print("\n" + "="*80)
print("URBAN vs RURAL SUMMARY")
print("="*80)

print("\nUrban (sharp boundaries):")
print(f"  Spline Advantage: {urban_stats['mean']:+.2f}% ± {urban_stats['std']:.2f}%")

print("\nRural (smooth variation):")
print(f"  Spline Advantage: {rural_stats['mean']:+.2f}% ± {rural_stats['std']:.2f}%")

# Test if urban and rural differ
t_stat, p_value = stats.ttest_ind(urban_adv, rural_adv)
print(f"\nUrban vs Rural Difference: t={t_stat:.3f}, p={p_value:.4f}")
if p_value < 0.05:
    print("  ✅ SIGNIFICANT: Urban/rural matters (p < 0.05)")
else:
    print("  ❌ NOT SIGNIFICANT: No urban/rural effect (p ≥ 0.05)")

print("="*80)

EXPERIMENT 4: URBAN vs RURAL REGIONS

Region: URBAN_EAST_US (urban)
Description: US East Coast (NYC, Boston, DC)
  Seed 42: Pop range [0.0, 11586.4] relu=0.6915 spline=0.7308 
  Seed 43: Pop range [0.0, 11586.4] relu=-0.0062 spline=-0.1601 
  Seed 44: Pop range [0.0, 6013.1] relu=-0.3711 spline=-0.6026 
  Seed 45: Pop range [0.0, 11586.4] relu=-0.7038 spline=-1.4757 
  Seed 46: Pop range [0.0, 11586.4] relu=-0.0960 spline=0.0147 
  Seed 47: Pop range [0.0, 11586.4] relu=-1.0538 spline=-0.5894 
  Seed 48: Pop range [0.0, 11586.4] relu=0.0108 spline=0.0088 
  Seed 49: Pop range [0.0, 6013.1] relu=-0.1044 spline=-0.4605 
  Seed 50: Pop range [0.0, 11586.4] relu=-0.0513 spline=0.1018 
  Seed 51: Pop range [0.0, 11586.4] relu=-1.0191 spline=-0.9159 

Region: RURAL_GREAT_PLAINS (rural)
Description: US Great Plains (sparse population)
  Seed 42: Pop range [0.0, 1846.2] relu=0.1570 spline=0.1147 
  Seed 43: Pop range [0.0, 1846.2] relu=-0.0142 spline=-0.0164 
  Seed 44: Pop range [0.0, 1846.2]

---
## Experiment 5: Extended Training (200 Epochs)

Same as NB21

In [13]:
print("="*80)
print("EXPERIMENT 5: CONVERGENCE ANALYSIS (200 Epochs)")
print("="*80)

SEEDS = range(42, 47)  # 5 seeds
ACTIVATIONS = ['relu', 'spline']
EPOCHS_CONV = 200

results_exp5 = []

for seed in SEEDS:
    print(f"\nSeed {seed}:")

    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        population, lons, lats, n_samples=15000, seed=seed
    )

    for act in ACTIVATIONS:
        print(f"  {act.upper()}...", end=" ")

        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_population_model(
            f'pop_conv_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            epochs=EPOCHS_CONV,
            verbose=False,
            track_epochs=True
        )

        res['seed'] = seed
        res['activation'] = act

        results_exp5.append(res)
        print(f"Best R²={res['r2']:.4f}")

print("\n" + "="*80)
print("CONVERGENCE SUMMARY")
print("="*80)

for act in ACTIVATIONS:
    act_results = [r for r in results_exp5 if r['activation'] == act]

    r2_at_100 = []
    r2_at_200 = []

    for res in act_results:
        history = res['r2_history']
        r2_at_100.append(max(history[:100]))
        r2_at_200.append(max(history))

    improvement = np.array(r2_at_200) - np.array(r2_at_100)
    imp_stats = compute_stats(improvement)

    print(f"\n{act.upper()}:")
    print(f"  R² at epoch 100: {np.mean(r2_at_100):.4f} ± {np.std(r2_at_100):.4f}")
    print(f"  R² at epoch 200: {np.mean(r2_at_200):.4f} ± {np.std(r2_at_200):.4f}")
    print(f"  Improvement:     {imp_stats['mean']:.4f} ± {imp_stats['std']:.4f}")

    if imp_stats['mean'] > 0.01:
        print("  ⚠️  UNDERTRAINING: Models still improving")
    else:
        print("  ✅ CONVERGED: 100 epochs sufficient")

print("="*80)

EXPERIMENT 5: CONVERGENCE ANALYSIS (200 Epochs)

Seed 42:
  RELU... Best R²=0.5717
  SPLINE... Best R²=0.5614

Seed 43:
  RELU... Best R²=0.6246
  SPLINE... Best R²=0.6222

Seed 44:
  RELU... Best R²=0.5602
  SPLINE... Best R²=0.5657

Seed 45:
  RELU... Best R²=0.6167
  SPLINE... Best R²=0.6063

Seed 46:
  RELU... Best R²=0.6096
  SPLINE... Best R²=0.5906

CONVERGENCE SUMMARY

RELU:
  R² at epoch 100: 0.5965 ± 0.0257
  R² at epoch 200: 0.5965 ± 0.0257
  Improvement:     0.0000 ± 0.0000
  ✅ CONVERGED: 100 epochs sufficient

SPLINE:
  R² at epoch 100: 0.5892 ± 0.0233
  R² at epoch 200: 0.5892 ± 0.0233
  Improvement:     0.0000 ± 0.0000
  ✅ CONVERGED: 100 epochs sufficient


---
## Final Summary: Population vs Elevation

Compare NB21 (elevation) and NB21b (population) findings

In [14]:
print("="*80)
print("FINAL SUMMARY: POPULATION REPRODUCIBILITY VALIDATION")
print("="*80)

print("\n" + "="*80)
print("EXPERIMENT 1: Global Multi-Seed (Population 15')")
print("="*80)
relu_r2s = df_exp1[df_exp1['activation'] == 'relu']['r2'].values
spline_r2s = df_exp1[df_exp1['activation'] == 'spline']['r2'].values
advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
adv_stats_exp1 = compute_stats(advantages)

print(f"Spline Advantage: {adv_stats_exp1['mean']:+.2f}% ± {adv_stats_exp1['std']:.2f}%")
print(f"95% CI: [{adv_stats_exp1['ci_low']:+.2f}%, {adv_stats_exp1['ci_high']:+.2f}%]")
print(f"CV: {adv_stats_exp1['cv']:.1f}%")

if adv_stats_exp1['cv'] < 20:
    print("\n✅ REPRODUCIBLE: Low variance (CV < 20%)")
elif adv_stats_exp1['cv'] < 50:
    print("\n⚠️  MODERATE VARIANCE: CV 20-50%")
else:
    print("\n❌ HIGH VARIANCE: CV > 50%, unreliable")

print("\n" + "="*80)
print("CROSS-TASK COMPARISON")
print("="*80)

print("\nTASK COMPARISON (use NB21 results):")
print("\nElevation (NB21):")
print("  Spline Advantage: [FROM NB21 RESULTS]")
print("\nPopulation (NB21b):")
print(f"  Spline Advantage: {adv_stats_exp1['mean']:+.2f}% ± {adv_stats_exp1['std']:.2f}%")

print("\n" + "="*80)
print("KEY QUESTIONS ANSWERED")
print("="*80)

print("\n1. Is population task reproducible?")
if adv_stats_exp1['cv'] < 20:
    print("   ✅ Yes - results stable across seeds")
else:
    print("   ❌ No - high variance like elevation")

print("\n2. Does resolution matter for population?")
print("   See Exp 3 results above")

print("\n3. Do urban boundaries help splines?")
print("   See Exp 4 results above")

print("\n4. Are population and elevation results consistent?")
print("   Compare NB21 and NB21b Exp 1 findings")

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

print("\nNext steps depend on cross-task consistency:")
print("\nIF NB21 and NB21b show SAME pattern:")
print("  → Strong generalizable finding")
print("  → Either 'ReLU always wins' or 'Splines always help'")
print("\nIF NB21 and NB21b DIFFER:")
print("  → Task-dependent effects exist")
print("  → Investigate what differs (smoothness, boundaries, etc.)")
print("\nIF BOTH show high variance:")
print("  → Fundamental reproducibility issue")
print("  → Focus on understanding why (NB22)")

print("\n" + "="*80)

FINAL SUMMARY: POPULATION REPRODUCIBILITY VALIDATION

EXPERIMENT 1: Global Multi-Seed (Population 15')
Spline Advantage: -0.23% ± 2.18%
95% CI: [-1.79%, +1.32%]
CV: -928.4%

✅ REPRODUCIBLE: Low variance (CV < 20%)

CROSS-TASK COMPARISON

TASK COMPARISON (use NB21 results):

Elevation (NB21):
  Spline Advantage: [FROM NB21 RESULTS]

Population (NB21b):
  Spline Advantage: -0.23% ± 2.18%

KEY QUESTIONS ANSWERED

1. Is population task reproducible?
   ✅ Yes - results stable across seeds

2. Does resolution matter for population?
   See Exp 3 results above

3. Do urban boundaries help splines?
   See Exp 4 results above

4. Are population and elevation results consistent?
   Compare NB21 and NB21b Exp 1 findings

RECOMMENDATION

Next steps depend on cross-task consistency:

IF NB21 and NB21b show SAME pattern:
  → Strong generalizable finding
  → Either 'ReLU always wins' or 'Splines always help'

IF NB21 and NB21b DIFFER:
  → Task-dependent effects exist
  → Investigate what differs (smoo

---
## Optional: Save to Google Drive

In [15]:
# Optional: Save results to Google Drive
if 'COLAB_GPU' in os.environ:
    save_dir = '/content/drive/MyDrive/learned_activation_results/nb21b/'
    os.makedirs(save_dir, exist_ok=True)

    df_exp1.to_csv(f'{save_dir}exp1_population_global_multiseed.csv', index=False)
    df_exp2.to_csv(f'{save_dir}exp2_population_sample_size.csv', index=False)
    df_exp3.to_csv(f'{save_dir}exp3_population_resolution.csv', index=False)
    df_exp4.to_csv(f'{save_dir}exp4_population_urban_rural.csv', index=False)

    print(f"✅ Results saved to Google Drive: {save_dir}")
else:
    print("Not running on Colab, skip Drive save")

✅ Results saved to Google Drive: /content/drive/MyDrive/learned_activation_results/nb21b/
